# Cross-Dataset Evaluation on CLEVR

Evaluate three VLMs — **SmolVLM2-2.2B**, **Qwen2-VL-2B**, and **InternVL3-2B** — on the
[CLEVR](https://cs.stanford.edu/people/jcjohns/clevr/) visual reasoning benchmark (Stanford / FAIR, CVPR 2017).

For each model we run:
1. **Zero-shot** evaluation (base pretrained weights)
2. **Fine-tuned** evaluation (LoRA checkpoints trained on PHLOP, if available)

This tests whether physics-oriented fine-tuning on PHLOP transfers to a different but related visual-reasoning domain.

In [ ]:
%pip install -q "transformers>=4.49.0" accelerate peft bitsandbytes
%pip install -q datasets huggingface_hub
%pip install -q decord Pillow matplotlib opencv-python-headless
%pip install -q qwen-vl-utils

## Configuration

In [ ]:
import os
import sys
import json
import gc
import time
import logging
import urllib.request
import zipfile
from pathlib import Path
from collections import Counter, defaultdict

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Suppress the "processor_kwargs" deprecation warning from transformers >= 5.4
logging.getLogger("transformers").setLevel(logging.ERROR)

CLEVR_DATA_DIR = Path("./clevr_data")
RESULTS_DIR = Path("./results/clevr")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES = 500          # set to None to evaluate ALL val questions (~150k)
MAX_NEW_TOKENS = 32

CHECKPOINTS_DIR = Path("./results/models")  # LoRA adapters from PHLOP fine-tuning (all models)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
dtype = torch.float16 if device == "mps" else torch.bfloat16

print(f"Device: {device}  |  dtype: {dtype}  |  MAX_SAMPLES: {MAX_SAMPLES}")

## 1. Download & Load CLEVR Dataset

In [ ]:
CLEVR_URL = "https://dl.fbaipublicfiles.com/clevr/CLEVR_v1.0.zip"
CLEVR_ROOT = CLEVR_DATA_DIR / "CLEVR_v1.0"

def download_clevr():
    """Download and extract CLEVR v1.0 (~18 GB) if not already present."""
    if (CLEVR_ROOT / "questions" / "CLEVR_val_questions.json").exists():
        print("CLEVR already downloaded.")
        return
    CLEVR_DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = CLEVR_DATA_DIR / "CLEVR_v1.0.zip"
    if not zip_path.exists():
        print(f"Downloading CLEVR v1.0 from {CLEVR_URL} ...")
        urllib.request.urlretrieve(CLEVR_URL, str(zip_path))
        print("Download complete.")
    print("Extracting ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(str(CLEVR_DATA_DIR))
    print("Done.")

download_clevr()

In [ ]:
def load_clevr_val(max_samples=None):
    """Load CLEVR validation questions and return a list of dicts."""
    qpath = CLEVR_ROOT / "questions" / "CLEVR_val_questions.json"
    with open(qpath) as f:
        data = json.load(f)
    questions = data["questions"]
    if max_samples:
        rng = np.random.RandomState(42)
        indices = rng.choice(len(questions), size=min(max_samples, len(questions)), replace=False)
        questions = [questions[i] for i in sorted(indices)]
    print(f"Loaded {len(questions)} CLEVR validation questions")
    return questions


def get_clevr_image(question_entry) -> Image.Image:
    """Load the image for a CLEVR question entry."""
    img_path = CLEVR_ROOT / "images" / "val" / question_entry["image_filename"]
    return Image.open(img_path).convert("RGB")


def get_clevr_image_path(question_entry) -> str:
    """Get the absolute image path for a CLEVR question entry."""
    return str((CLEVR_ROOT / "images" / "val" / question_entry["image_filename"]).resolve())


def clevr_question_type(entry) -> str:
    """Derive high-level question type from the CLEVR program."""
    program = entry.get("program", [])
    if not program:
        return "unknown"
    last_fn = program[-1].get("function", "") if program else ""
    type_map = {
        "exist": "exist",
        "count": "count",
        "equal_integer": "compare_integer",
        "less_than": "compare_integer",
        "greater_than": "compare_integer",
        "equal_size": "compare_attribute",
        "equal_color": "compare_attribute",
        "equal_material": "compare_attribute",
        "equal_shape": "compare_attribute",
        "query_size": "query_attribute",
        "query_color": "query_attribute",
        "query_material": "query_attribute",
        "query_shape": "query_attribute",
    }
    return type_map.get(last_fn, last_fn or "unknown")


clevr_questions = load_clevr_val(MAX_SAMPLES)

type_counts = Counter(clevr_question_type(q) for q in clevr_questions)
print("\nQuestion-type distribution:")
for qt, c in type_counts.most_common():
    print(f"  {qt}: {c}")

# --- Verify data format ---
print("\n--- Sample CLEVR questions (first 5) ---")
for q in clevr_questions[:5]:
    print(f"  Image: {q['image_filename']}")
    print(f"  Q: {q['question']}")
    print(f"  A: {q['answer']}  (type: {clevr_question_type(q)})")
    print()

# Verify images are accessible
sample_img = get_clevr_image(clevr_questions[0])
print(f"Sample image size: {sample_img.size}, mode: {sample_img.mode}")

# Show answer vocabulary
all_answers = Counter(str(q["answer"]).lower() for q in clevr_questions)
print(f"\nUnique answers ({len(all_answers)}): {dict(all_answers.most_common(20))}")

## 2. Common Evaluation Helpers

In [ ]:
CLEVR_VALID_ANSWERS = {
    "yes", "no",
    "0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
    "gray", "red", "blue", "green", "brown", "purple", "cyan", "yellow",
    "cube", "sphere", "cylinder",
    "small", "large",
    "metal", "rubber", "shiny", "matte",
}


def build_clevr_prompt(question: str) -> str:
    """Build a minimal VQA prompt for CLEVR."""
    return (
        "This image shows a 3D rendered scene with geometric objects "
        "(cubes, spheres, cylinders) of different colors, sizes, and materials.\n\n"
        f"Question: {question}\n\n"
        "Answer with ONLY one of the following:\n"
        "- For yes/no questions: \"yes\" or \"no\"\n"
        "- For counting questions: a single number (0-10)\n"
        "- For color questions: gray, red, blue, green, brown, purple, cyan, or yellow\n"
        "- For shape questions: cube, sphere, or cylinder\n"
        "- For size questions: small or large\n"
        "- For material questions: metal or rubber\n\n"
        "Answer:"
    )


def normalize_clevr_answer(answer: str) -> str:
    """Normalize a CLEVR answer for comparison."""
    a = str(answer).strip().lower()
    a = a.rstrip(".!,;")
    a = a.replace("there are ", "").replace("there is ", "")
    a = a.replace("the answer is ", "").replace("answer: ", "")

    if a in ("true",):
        a = "yes"
    if a in ("false", "not"):
        a = "no"

    # Handle verbose counting ("three" -> "3")
    word_to_num = {
        "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
        "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
    }
    if a in word_to_num:
        a = word_to_num[a]

    # Handle material synonyms
    if a == "shiny":
        a = "metal"
    if a == "matte":
        a = "rubber"

    # If model outputs a sentence, try to extract the valid answer token
    if a not in CLEVR_VALID_ANSWERS:
        tokens = a.split()
        for token in reversed(tokens):
            clean_tok = token.strip(".,!;:\"'()")
            if clean_tok in CLEVR_VALID_ANSWERS:
                a = clean_tok
                break
            if clean_tok in word_to_num:
                a = word_to_num[clean_tok]
                break

    return a.strip()


def score_clevr_results(results: list[dict]) -> dict:
    """Compute overall and per-type accuracy."""
    correct = sum(1 for r in results if r["correct"])
    total = len(results)
    overall = correct / total if total else 0.0

    by_type = defaultdict(lambda: {"correct": 0, "total": 0})
    for r in results:
        qt = r.get("question_type", "unknown")
        by_type[qt]["total"] += 1
        if r["correct"]:
            by_type[qt]["correct"] += 1

    per_type = {
        qt: {
            "accuracy": v["correct"] / v["total"] if v["total"] else 0.0,
            "correct": v["correct"],
            "total": v["total"],
        }
        for qt, v in sorted(by_type.items())
    }

    return {"overall_accuracy": overall, "correct": correct, "total": total, "per_type": per_type}


def print_clevr_metrics(metrics: dict, title: str):
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(f"  Overall accuracy: {metrics['overall_accuracy']:.4f} ({metrics['correct']}/{metrics['total']})")
    print("  Per question type:")
    for qt, v in metrics["per_type"].items():
        print(f"    {qt:25s}  {v['accuracy']:.4f}  ({v['correct']}/{v['total']})")


def save_clevr_results(results: list[dict], metrics: dict, model_name: str, variant: str):
    """Save predictions and metrics to JSON."""
    pred_path = RESULTS_DIR / f"{model_name}_{variant}_predictions.json"
    met_path = RESULTS_DIR / f"{model_name}_{variant}_metrics.json"
    with open(pred_path, "w") as f:
        json.dump(results, f, indent=2, default=str)
    with open(met_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"  Saved {len(results)} predictions -> {pred_path}")
    print(f"  Saved metrics -> {met_path}")

## 3. SmolVLM2-2.2B Evaluation

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
try:
    from transformers.video_utils import VideoMetadata
except ImportError:
    from dataclasses import dataclass, field

    @dataclass
    class VideoMetadata:
        total_num_frames: int = 0
        fps: float = 25.0
        duration: float = 0.0
        frames_indices: list = field(default_factory=list)

SMOLVLM_MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
SMOL_NUM_FRAMES = 8


def run_smolvlm_on_clevr(model, processor, questions, tag="zero_shot", num_frames=SMOL_NUM_FRAMES):
    """Run SmolVLM on CLEVR questions using video input (image repeated as frames)."""
    model.eval()
    results = []

    for i, q in enumerate(tqdm(questions, desc=f"SmolVLM {tag}")):
        image = get_clevr_image(q)
        prompt = build_clevr_prompt(q["question"])

        # Repeat the image as video frames to match the video training pipeline
        video_frames = [image.copy() for _ in range(num_frames)]
        video_meta = VideoMetadata(
            total_num_frames=num_frames,
            fps=25.0,
            duration=num_frames / 25.0,
            frames_indices=list(range(num_frames)),
        )

        text = (
            "You are a visual reasoning system.\n"
            "Watch the video and answer the question.\n\n"
            "<video>\n\n"
            f"{prompt}"
        )

        inputs = processor(
            text=text,
            videos=[[video_frames]],
            video_metadata=[[video_meta]],
            return_tensors="pt",
        ).to(device)

        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(dtype)

        with torch.no_grad():
            input_len = inputs["input_ids"].shape[1]
            output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            generated = output_ids[:, input_len:]

        pred_raw = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
        pred = normalize_clevr_answer(pred_raw)
        gt = normalize_clevr_answer(q["answer"])

        results.append({
            "idx": i,
            "image": q["image_filename"],
            "question": q["question"],
            "question_type": clevr_question_type(q),
            "true_answer": q["answer"],
            "prediction_raw": pred_raw,
            "prediction": pred,
            "correct": pred == gt,
        })

        del inputs, output_ids, generated
        if i % 50 == 0:
            if device == "cuda":
                torch.cuda.empty_cache()
            elif device == "mps":
                torch.mps.empty_cache()

    return results


# --- Zero-shot ---
print("Loading SmolVLM2 (zero-shot) ...")
smol_processor = AutoProcessor.from_pretrained(SMOLVLM_MODEL_ID)
smol_model = AutoModelForImageTextToText.from_pretrained(
    SMOLVLM_MODEL_ID, torch_dtype=dtype,
).to(device)

smol_zs_results = run_smolvlm_on_clevr(smol_model, smol_processor, clevr_questions, tag="zero_shot")
smol_zs_metrics = score_clevr_results(smol_zs_results)
print_clevr_metrics(smol_zs_metrics, "SmolVLM2 — Zero-Shot on CLEVR")
save_clevr_results(smol_zs_results, smol_zs_metrics, "smolvlm", "zero_shot")

In [ ]:
from peft import PeftModel


def find_checkpoints(prefix: str) -> list[tuple[str, str]]:
    """Find LoRA checkpoints matching a model prefix (e.g. 'smolvlm', 'qwen2vl', 'internvl3')."""
    found = []
    if not CHECKPOINTS_DIR.exists():
        return found
    for subdir in sorted(CHECKPOINTS_DIR.iterdir()):
        if not subdir.is_dir():
            continue
        if not (subdir / "adapter_config.json").exists():
            continue
        name_lower = subdir.name.lower()
        if prefix == "smolvlm":
            # SmolVLM checkpoints don't have a prefix — they're the ones without internvl3/llama3/qwen2vl
            if not any(name_lower.startswith(p) for p in ("internvl3", "llama3", "qwen2vl")):
                found.append((subdir.name, str(subdir)))
        elif name_lower.startswith(prefix):
            found.append((subdir.name, str(subdir)))
    return found


# --- SmolVLM Fine-tuned ---
smol_ft_all_metrics = {}
smol_checkpoints = find_checkpoints("smolvlm")

if smol_checkpoints:
    print(f"Found {len(smol_checkpoints)} SmolVLM fine-tuned checkpoints:")
    for name, path in smol_checkpoints:
        print(f"  {name}: {path}")

    # Load processor if not already in memory
    if "smol_processor" not in globals() or smol_processor is None:
        smol_processor = AutoProcessor.from_pretrained(SMOLVLM_MODEL_ID)

    for ckpt_name, ckpt_path in smol_checkpoints:
        print(f"\nEvaluating SmolVLM fine-tuned: {ckpt_name}")
        base_model = AutoModelForImageTextToText.from_pretrained(
            SMOLVLM_MODEL_ID, torch_dtype=dtype,
        )
        ft_model = PeftModel.from_pretrained(base_model, ckpt_path)
        ft_model = ft_model.merge_and_unload().to(device)

        ft_results = run_smolvlm_on_clevr(ft_model, smol_processor, clevr_questions, tag=f"finetuned_{ckpt_name}")
        ft_metrics = score_clevr_results(ft_results)
        print_clevr_metrics(ft_metrics, f"SmolVLM Fine-tuned ({ckpt_name}) on CLEVR")
        save_clevr_results(ft_results, ft_metrics, "smolvlm", f"finetuned_{ckpt_name}")
        smol_ft_all_metrics[ckpt_name] = ft_metrics

        del ft_model, base_model
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
        elif device == "mps":
            torch.mps.empty_cache()
else:
    print("No SmolVLM fine-tuned checkpoints found. Skipping.")

# Free SmolVLM
if "smol_model" in globals():
    del smol_model
if "smol_processor" in globals():
    del smol_processor
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()

## 4. Qwen2-VL-2B Evaluation

In [ ]:
from transformers import Qwen2VLForConditionalGeneration
import tempfile
import shutil

QWEN2_VL_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
QWEN_VIDEO_FPS = 2.0


def _create_temp_video_from_image(image: Image.Image, num_frames: int = 8, fps: int = 2) -> str:
    """Create a temporary mp4 video by repeating a single image as frames."""
    import cv2
    tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    tmp_path = tmp.name
    tmp.close()

    img_array = np.array(image)
    h, w = img_array.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(tmp_path, fourcc, fps, (w, h))
    for _ in range(num_frames):
        writer.write(cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR))
    writer.release()
    return tmp_path


def run_qwen2vl_on_clevr(model, processor, questions, tag="zero_shot", num_frames=8):
    """Run Qwen2-VL on CLEVR questions using video input (image repeated as frames)."""
    from qwen_vl_utils import process_vision_info

    model.eval()
    results = []

    for i, q in enumerate(tqdm(questions, desc=f"Qwen2-VL {tag}")):
        image = get_clevr_image(q)
        prompt = build_clevr_prompt(q["question"])

        # Create a temporary video from the image
        video_path = _create_temp_video_from_image(image, num_frames=num_frames, fps=int(QWEN_VIDEO_FPS))

        conv = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "video",
                        "video": f"file://{video_path}",
                        "fps": QWEN_VIDEO_FPS,
                        "min_pixels": 128 * 128,
                        "max_pixels": 256 * 256,
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(conv)
        inputs = processor(
            text=[text], images=image_inputs, videos=video_inputs,
            return_tensors="pt",
        ).to(device)

        if "pixel_values_videos" in inputs:
            inputs["pixel_values_videos"] = inputs["pixel_values_videos"].to(dtype)

        with torch.no_grad():
            input_len = inputs["input_ids"].shape[1]
            output_ids = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )
            generated = output_ids[:, input_len:]

        pred_raw = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
        pred = normalize_clevr_answer(pred_raw)
        gt = normalize_clevr_answer(q["answer"])

        results.append({
            "idx": i,
            "image": q["image_filename"],
            "question": q["question"],
            "question_type": clevr_question_type(q),
            "true_answer": q["answer"],
            "prediction_raw": pred_raw,
            "prediction": pred,
            "correct": pred == gt,
        })

        # Clean up temp video
        os.unlink(video_path)
        del inputs, output_ids, generated
        if i % 50 == 0:
            if device == "cuda":
                torch.cuda.empty_cache()
            elif device == "mps":
                torch.mps.empty_cache()

    return results


# --- Zero-shot ---
print("Loading Qwen2-VL-2B (zero-shot) ...")
qwen_processor = AutoProcessor.from_pretrained(QWEN2_VL_MODEL_ID)
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN2_VL_MODEL_ID, torch_dtype=dtype,
).to(device)

qwen_zs_results = run_qwen2vl_on_clevr(qwen_model, qwen_processor, clevr_questions, tag="zero_shot")
qwen_zs_metrics = score_clevr_results(qwen_zs_results)
print_clevr_metrics(qwen_zs_metrics, "Qwen2-VL-2B — Zero-Shot on CLEVR")
save_clevr_results(qwen_zs_results, qwen_zs_metrics, "qwen2vl", "zero_shot")

del qwen_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()

In [ ]:
# --- Qwen2-VL Fine-tuned ---
qwen_ft_all_metrics = {}
qwen_checkpoints = find_checkpoints("qwen2vl")

if qwen_checkpoints:
    print(f"Found {len(qwen_checkpoints)} Qwen2-VL fine-tuned checkpoints:")
    for name, path in qwen_checkpoints:
        print(f"  {name}: {path}")

    for ckpt_name, ckpt_path in qwen_checkpoints:
        print(f"\nEvaluating Qwen2-VL fine-tuned: {ckpt_name}")
        base_model = Qwen2VLForConditionalGeneration.from_pretrained(
            QWEN2_VL_MODEL_ID, torch_dtype=dtype,
        )
        ft_model = PeftModel.from_pretrained(base_model, ckpt_path)
        ft_model = ft_model.merge_and_unload().to(device)

        ft_results = run_qwen2vl_on_clevr(ft_model, qwen_processor, clevr_questions, tag=f"finetuned_{ckpt_name}")
        ft_metrics = score_clevr_results(ft_results)
        print_clevr_metrics(ft_metrics, f"Qwen2-VL Fine-tuned ({ckpt_name}) on CLEVR")
        save_clevr_results(ft_results, ft_metrics, "qwen2vl", f"finetuned_{ckpt_name}")
        qwen_ft_all_metrics[ckpt_name] = ft_metrics

        del ft_model, base_model
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
        elif device == "mps":
            torch.mps.empty_cache()
else:
    print("No Qwen2-VL fine-tuned checkpoints found. Skipping.")

# Free Qwen processor
if "qwen_processor" in globals():
    del qwen_processor
gc.collect()

## 5. InternVL3-2B Evaluation

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T

INTERNVL3_MODEL_ID = "OpenGVLab/InternVL3-2B"
INTERN_INPUT_SIZE = 448
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_internvl_transform(input_size=INTERN_INPUT_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if hasattr(img, "convert") else img),
        T.Resize((input_size, input_size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


INTERN_NUM_FRAMES = 8


def run_internvl3_on_clevr(model, tokenizer, questions, tag="zero_shot", num_frames=INTERN_NUM_FRAMES):
    """Run InternVL3 on CLEVR questions using video input (image repeated as frames)."""
    model.eval()
    transform = build_internvl_transform()
    results = []

    gen_config = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }

    for i, q in enumerate(tqdm(questions, desc=f"InternVL3 {tag}")):
        image = get_clevr_image(q)
        prompt = build_clevr_prompt(q["question"])

        # Repeat the image as video frames (N, 3, H, W)
        single_frame = transform(image)
        pixel_values = single_frame.unsqueeze(0).repeat(num_frames, 1, 1, 1).to(device=device, dtype=dtype)

        # Build prompt with frame markers (matching PHLOP training format)
        image_tokens = "".join(f"Frame{j+1}: <image>\n" for j in range(num_frames))
        question_str = image_tokens + prompt

        try:
            with torch.no_grad():
                result = model.chat(
                    tokenizer, pixel_values,
                    question=question_str,
                    generation_config=gen_config,
                    history=None,
                )
            response = result[0] if isinstance(result, tuple) else result
            pred_raw = str(response).strip()
        except Exception as e:
            print(f"  Error at idx {i}: {e}")
            pred_raw = "Error"

        pred = normalize_clevr_answer(pred_raw)
        gt = normalize_clevr_answer(q["answer"])

        results.append({
            "idx": i,
            "image": q["image_filename"],
            "question": q["question"],
            "question_type": clevr_question_type(q),
            "true_answer": q["answer"],
            "prediction_raw": pred_raw,
            "prediction": pred,
            "correct": pred == gt,
        })

        del pixel_values
        if i % 50 == 0:
            if device == "cuda":
                torch.cuda.empty_cache()
            elif device == "mps":
                torch.mps.empty_cache()

    return results


# --- Zero-shot ---
print("Loading InternVL3-2B (zero-shot) ...")
intern_tokenizer = AutoTokenizer.from_pretrained(INTERNVL3_MODEL_ID, trust_remote_code=True)
intern_model = AutoModel.from_pretrained(
    INTERNVL3_MODEL_ID, torch_dtype=dtype, trust_remote_code=True,
).to(device).eval()

intern_zs_results = run_internvl3_on_clevr(intern_model, intern_tokenizer, clevr_questions, tag="zero_shot")
intern_zs_metrics = score_clevr_results(intern_zs_results)
print_clevr_metrics(intern_zs_metrics, "InternVL3-2B — Zero-Shot on CLEVR")
save_clevr_results(intern_zs_results, intern_zs_metrics, "internvl3", "zero_shot")

del intern_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()

In [ ]:
# --- InternVL3 Fine-tuned ---
intern_ft_all_metrics = {}
intern_checkpoints = find_checkpoints("internvl3")

if intern_checkpoints:
    print(f"Found {len(intern_checkpoints)} InternVL3 fine-tuned checkpoints:")
    for name, path in intern_checkpoints:
        print(f"  {name}: {path}")

    for ckpt_name, ckpt_path in intern_checkpoints:
        print(f"\nEvaluating InternVL3 fine-tuned: {ckpt_name}")
        base_model = AutoModel.from_pretrained(
            INTERNVL3_MODEL_ID, torch_dtype=dtype, trust_remote_code=True,
        )
        ft_model = PeftModel.from_pretrained(base_model, ckpt_path)
        ft_model = ft_model.merge_and_unload().to(device).eval()

        ft_results = run_internvl3_on_clevr(ft_model, intern_tokenizer, clevr_questions, tag=f"finetuned_{ckpt_name}")
        ft_metrics = score_clevr_results(ft_results)
        print_clevr_metrics(ft_metrics, f"InternVL3 Fine-tuned ({ckpt_name}) on CLEVR")
        save_clevr_results(ft_results, ft_metrics, "internvl3", f"finetuned_{ckpt_name}")
        intern_ft_all_metrics[ckpt_name] = ft_metrics

        del ft_model, base_model
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
        elif device == "mps":
            torch.mps.empty_cache()
else:
    print("No InternVL3 fine-tuned checkpoints found. Skipping.")

# Free InternVL3 tokenizer
if "intern_tokenizer" in globals():
    del intern_tokenizer
gc.collect()

## 6. Results Comparison

In [ ]:
import matplotlib.pyplot as plt

all_results = {}

# Zero-shot results (use .get() in case a model was skipped)
if "smol_zs_metrics" in globals():
    all_results["SmolVLM2 (zero-shot)"] = smol_zs_metrics
if "qwen_zs_metrics" in globals():
    all_results["Qwen2-VL (zero-shot)"] = qwen_zs_metrics
if "intern_zs_metrics" in globals():
    all_results["InternVL3 (zero-shot)"] = intern_zs_metrics

# Fine-tuned results
if "smol_ft_all_metrics" in globals():
    for ckpt_name, met in smol_ft_all_metrics.items():
        all_results[f"SmolVLM2 (ft: {ckpt_name})"] = met
if "qwen_ft_all_metrics" in globals():
    for ckpt_name, met in qwen_ft_all_metrics.items():
        all_results[f"Qwen2-VL (ft: {ckpt_name})"] = met
if "intern_ft_all_metrics" in globals():
    for ckpt_name, met in intern_ft_all_metrics.items():
        all_results[f"InternVL3 (ft: {ckpt_name})"] = met

print(f"Comparing {len(all_results)} model variants")

# --- Summary Table ---
print(f"\n{'Model':<35s} {'Overall':>10s}  ", end="")
all_qtypes = sorted({qt for m in all_results.values() for qt in m["per_type"]})
for qt in all_qtypes:
    print(f"{qt:>18s}", end="")
print()
print("-" * (35 + 12 + 18 * len(all_qtypes)))

for model_name, met in all_results.items():
    print(f"{model_name:<35s} {met['overall_accuracy']:>10.4f}  ", end="")
    for qt in all_qtypes:
        acc = met["per_type"].get(qt, {}).get("accuracy", 0.0)
        print(f"{acc:>18.4f}", end="")
    print()

# --- Bar Chart ---
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(all_results.keys())
accuracies = [all_results[m]["overall_accuracy"] for m in model_names]
colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

bars = ax.bar(range(len(model_names)), accuracies, color=colors)
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Accuracy")
ax.set_title("CLEVR Validation Accuracy — Zero-Shot vs. Fine-Tuned")
ax.set_ylim(0, 1.0)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.2%}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / "clevr_comparison.png"), dpi=150)
plt.show()
print(f"Saved chart to {RESULTS_DIR / 'clevr_comparison.png'}")

In [ ]:
# --- Per question-type grouped bar chart ---
fig, ax = plt.subplots(figsize=(14, 5))

n_models = len(all_results)
n_types = len(all_qtypes)
bar_width = 0.8 / n_models

for j, (model_name, met) in enumerate(all_results.items()):
    positions = [i + j * bar_width for i in range(n_types)]
    accs = [met["per_type"].get(qt, {}).get("accuracy", 0.0) for qt in all_qtypes]
    ax.bar(positions, accs, width=bar_width, label=model_name, color=colors[j])

ax.set_xticks([i + bar_width * (n_models - 1) / 2 for i in range(n_types)])
ax.set_xticklabels(all_qtypes, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Accuracy")
ax.set_title("CLEVR Accuracy by Question Type")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / "clevr_per_type_comparison.png"), dpi=150)
plt.show()
print(f"Saved chart to {RESULTS_DIR / 'clevr_per_type_comparison.png'}")

In [ ]:
summary = {
    "dataset": "CLEVR v1.0 (validation)",
    "n_samples": MAX_SAMPLES or "all",
    "results": {
        model_name: {
            "overall_accuracy": met["overall_accuracy"],
            "n_correct": met["correct"],
            "n_total": met["total"],
        }
        for model_name, met in all_results.items()
    },
}

summary_path = RESULTS_DIR / "clevr_evaluation_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nFull summary saved to {summary_path}")